Data Ingestion 

Langchain Document Structure

Core concepts: 
1. page_content(str)
2. metadata(dict)



In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="This is the content of the document. I am using to create RAG system.", 
    metadata={"source": "example.txt",
              "author": "Karmabir Chakraborty", 
              "data_created": "2026-04-12"
              }
            )
doc

Document(metadata={'source': 'example.txt', 'author': 'Karmabir Chakraborty', 'data_created': '2026-04-12'}, page_content='This is the content of the document. I am using to create RAG system.')

In [3]:
## Create a simple txt file

import os
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
sample_texts = {
    "../data/text_files/file1.txt": "This is the content of file 1. It contains information about Python programming.",
    "../data/text_files/file2.txt": "This is the content of file 2. It contains information about machine learning."
}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


In [5]:
## TextLoader

from langchain_community.document_loaders import TextLoader

/Users/karmabirchakraborty/Documents/vscodeprojects/study/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [6]:
loader = TextLoader("../data/text_files/file1.txt", encoding="utf-8")
loader

In [7]:
document = loader.load()

In [8]:
print(document)

[Document(metadata={'source': '../data/text_files/file1.txt'}, page_content='This is the content of file 1. It contains information about Python programming.')]


In [9]:
### Directory Loader

from langchain_community.document_loaders import DirectoryLoader
directory_loader = DirectoryLoader("../data/text_files", glob="*.txt",  
                                   loader_cls=TextLoader, 
                                   loader_kwargs={"encoding": "utf-8"}, 
                                   show_progress=True)

In [10]:
documents = directory_loader.load()
print(documents)

100%|██████████| 2/2 [00:00<00:00, 3975.64it/s]

[Document(metadata={'source': '../data/text_files/file2.txt'}, page_content='This is the content of file 2. It contains information about machine learning.'), Document(metadata={'source': '../data/text_files/file1.txt'}, page_content='This is the content of file 1. It contains information about Python programming.')]


In [11]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

/Users/karmabirchakraborty/Documents/vscodeprojects/study/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
from langchain_community.document_loaders import DirectoryLoader

directory_loader = DirectoryLoader(
    "../data/pdf_files", 
    glob = "**/*.pdf",
    loader_cls = PyMuPDFLoader,
    show_progress = True
)

In [13]:
pdf_documents = directory_loader.load()
print(pdf_documents)

100%|██████████| 18/18 [00:02<00:00,  6.42it/s]

[Document(metadata={'producer': 'pdfTeX-1.40.20; modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': 'IEEE Conference eXpress', 'creationdate': '2021-09-24T12:19:52-04:00', 'source': '../data/pdf_files/Transferability Analysis of an Adversarial Attack on.pdf', 'file_path': '../data/pdf_files/Transferability Analysis of an Adversarial Attack on.pdf', 'total_pages': 7, 'format': 'PDF 1.6', 'title': 'Transferability Analysis of an Adversarial Attack on Gender Classification to Face Recognition', 'author': 'Zohra Rezgui', 'subject': '2021 International Conference of the Biometrics Special Interest Group (BIOSIG);2021; ; ;10.1109/BIOSIG52210.2021.9548308', 'keywords': 'Transferability,adversarial attacks,gender classification,face recognition', 'moddate': '2022-08-24T21:18:20-04:00', 'trapped': '', 'modDate': "D:20220824211820-04'00'", 'creationDate': "D:20210924121952-04'00'", 'page': 0}, page_content='Transferability Analysis of an Adversarial Attack on\nGend

In [14]:
### Embeddings and Vector Store

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [16]:
class EmbeddingManager: 
    """Handles document embedding generation using SentenceTransformers"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager with a specified model.
        
        Args:
            model_name (str): The name of the SentenceTransformer model to use for generating embeddings.
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise
    
    def generate_embeddings(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """Generate embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of strings to generate embeddings for.
            batch_size (int): Batch size for encoding (smaller batch size = more stable).
        
        Returns: 
            numpy array of embeddings with shape (len(texts), embedding_dimension)
        """

        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.")
        
        print(f"Generate embeddings for {len(texts)} texts using model '{self.model_name}'...")
        print(f"Using batch size: {batch_size}")
        
        # Use smaller batch size for more stability
        embeddings = self.model.encode(
            texts, 
            show_progress_bar=True,
            batch_size=batch_size,
            convert_to_numpy=True
        )
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
        
## initialize embedding manager
embedding_manager = EmbeddingManager()

Model 'all-MiniLM-L6-v2' loaded successfully. Embedding dimension: 384


In [17]:
embedding_manager

In [19]:
# vectorstore

In [20]:
class VectorStore: 
    """Manages document embeddings and metadata, allowing for efficient similarity search."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store with an embedding manager.
        
        Args:
            embedding_manager (EmbeddingManager): An instance of the EmbeddingManager class to generate embeddings.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        import os
        """Initialize the ChromaDB client and collection."""
        try: 
            # Create ChromaDB client with persistence
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Create or get collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={'description': "PDF embeddings for RAG"}
            )
            print(f"Vector store initialized with collection '{self.collection_name}' at '{self.persist_directory}'")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray, batch_size: int = 4000):
        """Add documents and their embeddings to the vector store.
        
        Args:
            documents (List[Any]): A list of document objects containing metadata.
            embeddings (np.ndarray): A numpy array of shape (len(documents), embedding_dimension) containing the corresponding embeddings.
            batch_size (int): Max batch size for adding to ChromaDB (default 4000 to stay under limit).
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to vector store collection '{self.collection_name}'...")
        print(f"Using batch size: {batch_size}")

        # Process in batches to avoid ChromaDB batch size limits
        total_added = 0
        for batch_start in range(0, len(documents), batch_size):
            batch_end = min(batch_start + batch_size, len(documents))
            batch_docs = documents[batch_start:batch_end]
            batch_embeddings = embeddings[batch_start:batch_end]
            
            print(f"  Processing batch {batch_start // batch_size + 1}: documents {batch_start}-{batch_end}")
            
            # Prepare data for insertion
            ids = []
            metadatas = []
            document_texts = []
            embeddings_list = []

            for i, (doc, embedding) in enumerate(zip(batch_docs, batch_embeddings)):
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{batch_start + i}"
                ids.append(doc_id)

                # Prepare metadata as a dictionary - clean encoding
                metadata = {}
                for key, value in doc.metadata.items():
                    # Convert to string and clean encoding
                    if value is not None:
                        str_val = str(value)
                        # Remove surrogate characters
                        str_val = str_val.encode('utf-8', 'ignore').decode('utf-8')
                        metadata[key] = str_val
                
                metadata['source'] = str(batch_start + i)
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)

                # Document content - clean encoding
                doc_text = str(doc.page_content)
                doc_text = doc_text.encode('utf-8', 'ignore').decode('utf-8')
                document_texts.append(doc_text)

                # Embedding
                embeddings_list.append(embedding.tolist())

            # Add batch to ChromaDB collection
            try:
                self.collection.add(
                    ids=ids,
                    metadatas=metadatas,
                    documents=document_texts,
                    embeddings=embeddings_list
                )
                total_added += len(batch_docs)
                print(f"    ✓ Added {len(batch_docs)} documents (total: {total_added})")

            except Exception as e:
                print(f"    ✗ Error adding batch: {e}")
                raise
        
        print(f"Successfully added {total_added} documents to vector store.")
        print(f"Total documents in collection: {self.collection.count()}")

# Create VectorStore instance
vectorstore = VectorStore()

Vector store initialized with collection 'pdf_documents' at '../data/vector_store'
Existing documents in collection: 11054


In [21]:
from pathlib import Path
def process_all_pdfs(pdf_directory: str = "../data/pdf_files"):
    """Process all PDF files in the specified directory"""
    all_documents = []
    pdf_directory_path = Path(pdf_directory)

    # Find all PDF files in the directory
    pdf_files = list(pdf_directory_path.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 18 PDF files to process

Processing: Transferability Analysis of an Adversarial Attack on.pdf
  ✓ Loaded 7 pages

Processing: Enhancing Adversarial Attacks The Similar Target.pdf
  ✓ Loaded 9 pages

Processing: preprints202302.0004.v1.pdf
  ✓ Loaded 18 pages

Processing: A Survey on Transferability of Adversarial Examples Across.pdf
  ✓ Loaded 35 pages

Processing: Transferability Analysis of an Adversarial Attack.pdf
  ✓ Loaded 15 pages

Processing: Adversarial Attacks and Defenses in Deep Learning.pdf
  ✓ Loaded 39 pages

Processing: Extracting Trading Signals from Limit Order Book Market Data.pdf
  ✓ Loaded 8 pages

Processing: ROOM.pdf
  ✓ Loaded 12 pages

Processing: ATTACKING DEEP NETWORKS WITH SURROGATEBASED.pdf
  ✓ Loaded 17 pages

Processing: Detecting Adversarial Examples Using Surrogate Models.pdf
  ✓ Loaded 30 pages

Processing: Efficient Black-Box Adversarial Attacks.pdf
  ✓ Loaded 525 pages

Processing: Improving Adversarial Training using Vulnerability-Aware.pdf
  

In [22]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=500, chunk_overlap=50):
    """ Split documents into smaller chunks
    Args:
        documents: List of Document objects or raw strings.
        chunk_size: Max characters per chunk.
        chunk_overlap: Overlap between chunks.
    Returns:
        List of Document chunks
    """
    # Ensure all inputs are Document objects
    if isinstance(documents[0], str):
        documents = [Document(page_content=doc, metadata={}) for doc in documents]

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # show example of chunk
    if split_docs:
        print(f"\n Example chunk")
        print(f"Content : {split_docs[0].page_content[:200]}...")
        print(f"Metadata : {split_docs[0].metadata}")
    
    return split_docs


chunks = split_documents(all_pdf_documents)
print(f"Total chunks created: {len(chunks)}")

Split 804 documents into 5527 chunks

 Example chunk
Content : Transferability Analysis of an Adversarial Attack on
Gender Classiﬁcation to Face Recognition
Zohra Rezgui 1, Amina Bassit 1,2
{z.rezgui, a.bassit}@utwente.nl,
1 DMB Group and 2 SCS Group, University ...
Metadata : {'producer': 'pdfTeX-1.40.20; modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': 'IEEE Conference eXpress', 'creationdate': '2021-09-24T12:19:52-04:00', 'trapped': '/False', 'moddate': '2022-08-24T21:18:20-04:00', 'ieee issue id': '9548283', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019/W32TeX) kpathsea version 6.3.1', 'ieee publication id': '9547357', 'title': 'Transferability Analysis of an Adversarial Attack on Gender Classification to Face Recognition', 'meeting ending date': '17 Sept. 2021', 'keywords': 'Transferability,adversarial attacks,gender classification,face recognition', 'ieee article id': '9548308', 'subject': '2021 International Co

In [23]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.20; modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': 'IEEE Conference eXpress', 'creationdate': '2021-09-24T12:19:52-04:00', 'trapped': '/False', 'moddate': '2022-08-24T21:18:20-04:00', 'ieee issue id': '9548283', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019/W32TeX) kpathsea version 6.3.1', 'ieee publication id': '9547357', 'title': 'Transferability Analysis of an Adversarial Attack on Gender Classification to Face Recognition', 'meeting ending date': '17 Sept. 2021', 'keywords': 'Transferability,adversarial attacks,gender classification,face recognition', 'ieee article id': '9548308', 'subject': '2021 International Conference of the Biometrics Special Interest Group (BIOSIG);2021; ; ;10.1109/BIOSIG52210.2021.9548308', 'meeting starting date': '15 Sept. 2021', 'author': 'Zohra Rezgui', 'source': '../data/pdf_files/Transferability Analysis of an Adversarial Attack on.pdf', 

In [24]:
### Convert the text to embeddings 

# Filter, clean and validate texts
texts = []
for doc in chunks:
    if doc.page_content:
        # Convert to string and strip whitespace
        text = str(doc.page_content).strip()
        
        # Remove problematic characters and ensure proper encoding
        # Keep only valid UTF-8 characters
        text = text.encode('utf-8', errors='ignore').decode('utf-8')
        
        # Skip very short texts
        if len(text) > 10:
            texts.append(text)

print(f"Valid texts: {len(texts)} out of {len(chunks)} chunks")
print(f"Texts removed: {len(chunks) - len(texts)}")
if texts:
    print(f"First text sample: {texts[0][:100]}...")
    print(f"First text length: {len(texts[0])}")
texts[:5]  # Show first 5

Valid texts: 5527 out of 5527 chunks
Texts removed: 0
First text sample: Transferability Analysis of an Adversarial Attack on
Gender Classiﬁcation to Face Recognition
Zohra ...
First text length: 493


['Transferability Analysis of an Adversarial Attack on\nGender Classiﬁcation to Face Recognition\nZohra Rezgui 1, Amina Bassit 1,2\n{z.rezgui, a.bassit}@utwente.nl,\n1 DMB Group and 2 SCS Group, University of Twente, Enschede, The Netherlands\nAbstract—Modern biometric systems establish their decision\nbased on the outcome of machine learning (ML) classiﬁers trained\nto make accurate predictions. Such classiﬁers are vulnerable to\ndiverse adversarial attacks, altering the classiﬁers’ predictions by',
 'adding a crafted perturbation. According to ML literature, those\nattacks are transferable among models that perform the same\ntask. However, models performing different tasks, but sharing the\nsame input space and the same model architecture, were never\nincluded in transferability scenarios. In this paper, we analyze\nthis phenomenon for the special case of VGG16-based biometric\nclassiﬁers. Concretely, we study the effect of the white-box',
 'FGSM attack, on a gender classiﬁer and com

In [25]:
### Generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

### Store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generate embeddings for 5527 texts using model 'all-MiniLM-L6-v2'...
Using batch size: 32


Batches: 100%|██████████| 173/173 [00:10<00:00, 16.95it/s]


Generated embeddings with shape: (5527, 384)
Adding 5527 documents to vector store collection 'pdf_documents'...
Using batch size: 4000
  Processing batch 1: documents 0-4000
    ✓ Added 4000 documents (total: 4000)
  Processing batch 2: documents 4000-5527
    ✓ Added 1527 documents (total: 5527)
Successfully added 5527 documents to vector store.
Total documents in collection: 16581


In [26]:
## RAG RETRIEVAL PIPELINE

In [27]:
class RAGRetrieval:
    """Manages RAG retrieval pipeline for document search"""
    
    def __init__(self, vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAG retrieval pipeline with a vector store and embedding manager.
        Args:
        vectorstore (VectorStore): An instance of the VectorStore class for managing document embeddings.
        embedding_manager (EmbeddingManager): An instance of the EmbeddingManager class for generating embeddings.
        """
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.3) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents from the vector store based on a query.
        Args:
        query (str): The input query string for which to retrieve relevant documents.
        top_k (int): The maximum number of top results to return (default is 5).
        score_threshold (float): Minimum similarity score for a document to be considered relevant (default is 0.3, adjusted for L2 distance).
        Returns:
        List[Dict[str, Any]]: A list of dictionaries containing the retrieved documents and their metadata.
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}\n")

        try:
            # Generate embedding for the query
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]

            # Search in the vector store
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=['documents', 'metadatas', 'distances']
            )

            retrieved_docs = []

            # Check if results exist and have documents
            if not results or not results.get('documents'):
                print("No documents found for this query.")
                return retrieved_docs
            
            # Get the first batch of results (ChromaDB returns list of lists)
            documents_batch = results.get('documents', [])
            if not documents_batch or not documents_batch[0]:
                print("No documents found for this query.")
                return retrieved_docs

            documents = documents_batch[0] if documents_batch else []
            metadatas = results.get('metadatas', [[]])[0] if results.get('metadatas') else []
            distances = results.get('distances', [[]])[0] if results.get('distances') else []
            ids = results.get('ids', [[]])[0] if results.get('ids') else []

            print(f"Found {len(documents)} documents\n")

            for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                # For L2 distance: lower distance = more similar. Invert and normalize
                # Using 1 / (1 + distance) to get a similarity score between 0 and 1
                similarity_score = 1 / (1 + distance) if distance >= 0 else 0
                
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'id': doc_id,
                        'document': document[:200] + "..." if len(document) > 200 else document,
                        'full_document': document,
                        'metadata': metadata,
                        'similarity_score': similarity_score,
                        'rank': len(retrieved_docs) + 1
                    })
                    print(f"{len(retrieved_docs)}. ID: {doc_id}")
                    print(f"   Similarity: {similarity_score:.4f} (distance: {distance:.4f})")
                    print(f"   Content: {document[:100]}...\n")
                else:
                    print(f"Skipping document ID: {doc_id} due to low similarity ({similarity_score:.4f})\n")

            print(f"✓ Retrieved {len(retrieved_docs)} relevant documents")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {str(e)}")
            import traceback
            traceback.print_exc()
            return []
        
rag_retriever = RAGRetrieval(vectorstore, embedding_manager)

In [28]:
rag_retriever

In [29]:
rag_retriever.retrieve("What is FGSM?", top_k=3, score_threshold=0.5)

Retrieving documents for query: 'What is FGSM?'
Top K: 3, Score Threshold: 0.5

Generate embeddings for 1 texts using model 'all-MiniLM-L6-v2'...
Using batch size: 32


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.85it/s]

Generated embeddings with shape: (1, 384)
Found 3 documents

1. ID: doc_f9ad6be3_3297
   Similarity: 0.5005 (distance: 0.9980)
   Content: the FGSM attack.
ALD
2. Compared to the aforementioned three metrics, its variation with respect
to ...

2. ID: doc_62e3df88_3297
   Similarity: 0.5005 (distance: 0.9980)
   Content: the FGSM attack.
ALD
2. Compared to the aforementioned three metrics, its variation with respect
to ...

3. ID: doc_15e30824_3297
   Similarity: 0.5005 (distance: 0.9980)
   Content: the FGSM attack.
ALD
2. Compared to the aforementioned three metrics, its variation with respect
to ...

✓ Retrieved 3 relevant documents


[{'id': 'doc_f9ad6be3_3297',
  'document': 'the FGSM attack.\nALD\n2. Compared to the aforementioned three metrics, its variation with respect\nto the perturbation budget is more consistent, meaning the curve of this metric\nis closer to a straight...',
  'full_document': 'the FGSM attack.\nALD\n2. Compared to the aforementioned three metrics, its variation with respect\nto the perturbation budget is more consistent, meaning the curve of this metric\nis closer to a straight line.\nMSE . The diﬀerence in MSE between the model groups increases with the\nperturbation size. However, under the FGSM attack, theMSE of the Self Adap-\ntive clean model is lower than that of all models. For the other adversarial attack',
  'metadata': {'total_pages': '525',
   'source': '3297',
   'moddate': '2024-09-24T02:08:25-04:00',
   'content_length': 437,
   'creationdate': '2024-02-24T18:30:04+05:30',
   'creator': 'PyPDF',
   'page_label': '250',
   'source_file': 'Efficient Black-Box Adversarial Attack

In [30]:
# Test with different queries and thresholds
results = rag_retriever.retrieve("What is ROOM?", top_k=5, score_threshold=0.3)

Retrieving documents for query: 'What is ROOM?'
Top K: 5, Score Threshold: 0.3

Generate embeddings for 1 texts using model 'all-MiniLM-L6-v2'...
Using batch size: 32


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.42it/s]

Generated embeddings with shape: (1, 384)
Found 5 documents

1. ID: doc_eeb9e98e_1354
   Similarity: 0.4476 (distance: 1.2344)
   Content: spective for a given noise budget. We want to answer whether
ROOM’s impact is inherent or can be ach...

2. ID: doc_03650032_1354
   Similarity: 0.4476 (distance: 1.2344)
   Content: spective for a given noise budget. We want to answer whether
ROOM’s impact is inherent or can be ach...

3. ID: doc_7c1bb547_1354
   Similarity: 0.4476 (distance: 1.2344)
   Content: spective for a given noise budget. We want to answer whether
ROOM’s impact is inherent or can be ach...

4. ID: doc_86f38fa1_1385
   Similarity: 0.4419 (distance: 1.2629)
   Content: eration. Finally, we would also like to explore whether the
ROOM strategy results in different oppor...

5. ID: doc_bcc1d12b_1385
   Similarity: 0.4419 (distance: 1.2629)
   Content: eration. Finally, we would also like to explore whether the
ROOM strategy results in different oppor...

✓ Retrieved 5 relevant d

In [31]:
# Integration Vectordb Context Pipeline with LLM Output Generation

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.environ.get("GROQ_API_KEY")

In [35]:
print(f"GROQ API Key: {'Set' if groq_api_key else 'Not Set'}")

GROQ API Key: Set


In [48]:
llm = ChatGroq(groq_api_key = groq_api_key, model="llama-3.1-8b-instant", temperature=0.7, max_tokens=1024)

In [49]:
# Simple RAG Function
def rag_simple(query, retreiver, llm, top_k=3):
    ## retrieve relevant documents
    results = retreiver.retrieve(query, top_k=top_k, score_threshold=0.3)
    context = "\n\n".join([doc['full_document'] for doc in results]) if results else ""
    if not context:
        print("No relevant documents found. Returning default response.")
        return "I'm sorry, I couldn't find any relevant information to answer your question."

    ## Generate the answer using the retrieved context
    prompt = f"Answer the following question based on the provided context:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content




In [50]:
answer = rag_simple("What is FGSM?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is FGSM?'
Top K: 3, Score Threshold: 0.3

Generate embeddings for 1 texts using model 'all-MiniLM-L6-v2'...
Using batch size: 32


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.95it/s]

Generated embeddings with shape: (1, 384)
Found 3 documents

1. ID: doc_f9ad6be3_3297
   Similarity: 0.5005 (distance: 0.9980)
   Content: the FGSM attack.
ALD
2. Compared to the aforementioned three metrics, its variation with respect
to ...

2. ID: doc_62e3df88_3297
   Similarity: 0.5005 (distance: 0.9980)
   Content: the FGSM attack.
ALD
2. Compared to the aforementioned three metrics, its variation with respect
to ...

3. ID: doc_15e30824_3297
   Similarity: 0.5005 (distance: 0.9980)
   Content: the FGSM attack.
ALD
2. Compared to the aforementioned three metrics, its variation with respect
to ...

✓ Retrieved 3 relevant documents


FGSM stands for Fast Gradient Sign Method. It is a type of adversarial attack used to manipulate machine learning models by adding small perturbations to the input data.


In [51]:
## Enhanced RAG Function with better prompt and error handling

In [53]:
def rag_advanced(query, retriever, llm, top_k = 5, min_score = 0.2, return_context = False):
    """Enhanced RAG function with improved prompt and error handling
    Args:
        query: The input question or query string.
        retriever: An instance of the RAGRetrieval class to perform document retrieval.
        llm: An instance of a language model (e.g., ChatGroq) to generate answers.
        top_k: The number of top relevant documents to retrieve (default is 5).
        min_score: Minimum similarity score for a document to be considered relevant (default is 0.2).
        return_context: Whether to return the retrieved context along with the answer (default is False).
    Returns:
        The generated answer from the LLM, and optionally the retrieved context.
    """
    results  = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': "I'm sorry, I couldn't find any relevant information to answer your question.", 'context': []}
    
    # Prepare context and sources
    context = "\n\n".join([doc['full_document'] for doc in results])
    sources = [{
        'sources': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'similarity_score': doc['similarity_score'],
        'preview': doc['document'][:120] + "..."
    } for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)

    # Generate answer
    prompt = f"""Use the following context to answer the question. If the context is not relevant, say you don't know. Always cite your sources from the context. \n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

result = rag_advanced("What is FGSM?", rag_retriever, llm, top_k=5, min_score=0.2, return_context=True)
print("Answer:", result['answer'])
print("\nSources:", result ['sources'] )
print("\nConfidence:", result['confidence'])
print("\nContext:", result.get('context', 'No context returned'))



Retrieving documents for query: 'What is FGSM?'
Top K: 5, Score Threshold: 0.2

Generate embeddings for 1 texts using model 'all-MiniLM-L6-v2'...
Using batch size: 32


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.93it/s]

Generated embeddings with shape: (1, 384)
Found 5 documents

1. ID: doc_f9ad6be3_3297
   Similarity: 0.5005 (distance: 0.9980)
   Content: the FGSM attack.
ALD
2. Compared to the aforementioned three metrics, its variation with respect
to ...

2. ID: doc_62e3df88_3297
   Similarity: 0.5005 (distance: 0.9980)
   Content: the FGSM attack.
ALD
2. Compared to the aforementioned three metrics, its variation with respect
to ...

3. ID: doc_15e30824_3297
   Similarity: 0.5005 (distance: 0.9980)
   Content: the FGSM attack.
ALD
2. Compared to the aforementioned three metrics, its variation with respect
to ...

4. ID: doc_127139ef_620
   Similarity: 0.4840 (distance: 1.0662)
   Content: Published in Transactions on Machine Learning Research (05 /2024)
Junhua Zou, Yexin Duan, Boyu Li, W...

5. ID: doc_8b1c4c49_620
   Similarity: 0.4840 (distance: 1.0662)
   Content: Published in Transactions on Machine Learning Research (05 /2024)
Junhua Zou, Yexin Duan, Boyu Li, W...

✓ Retrieved 5 relevant doc

Answer: Based on the given context, FGSM stands for the Fast Gradient Sign Method, but it is not explicitly stated in the provided text. 

However, a source in the context (Junhua Zou, Yexin Duan, Boyu Li, Wu Zhang, Yu Pan, and Zhisong Pan. Making adversarial examples more transferable and indistinguishable. In Proceedings of the AAAI Conference on Artificial Intelligence, pp. 3662–3670, 2022) mentions the FGSM attack, suggesting that FGSM is indeed an attack. 

For the FGSM attack specifics, I do not know from the given context as it does not provide detailed information about the FGSM attack.

Sources: [{'sources': 'Efficient Black-Box Adversarial Attacks.pdf', 'page': '270', 'similarity_score': 0.5005008225326942, 'preview': 'the FGSM attack.\nALD\n2. Compared to the aforementioned three metrics, its variation with respect\nto the perturbation bud...'}, {'sources': 'Efficient Black-Box Adversarial Attacks.pdf', 'page': '270', 'similarity_score': 0.5005008225326942, 'preview': 'the F